# UCS420: Cognitive Computing
## Assignment 4 - A Cognitive FAQ System Using Pandas (Nova 2.0)

Roll Number: **1024160106**

## Q1. Build Your Personalized Knowledge Base

Roll number = `1024160106` -> last two digits = **0** and **6**.

`category = ["billing", "account", "general"][d % 3]`

- digit `0` -> `0 % 3 = 0` -> **billing**
- digit `6` -> `6 % 3 = 0` -> **billing**

Both digits map to the *billing* category, so two distinct, realistic billing-themed FAQ entries were invented for them.

In [1]:
import pandas as pd

roll_number = "1024160106"

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# Personalized entries derived from roll number digits "0" and "6"
personalized_entries = [
    # digit 0 -> 0 % 3 = 0 -> billing
    {"question": "why was i charged twice this month", "answer": "Duplicate charges are refunded within 3-5 business days.",
     "keywords": "double charge refund duplicate", "category": "billing"},
    # digit 6 -> 6 % 3 = 0 -> billing
    {"question": "can i get an invoice for my last payment", "answer": "Yes, invoices are emailed automatically and available under Billing > Invoices.",
     "keywords": "invoice receipt payment billing", "category": "billing"},
]

faq_df = pd.DataFrame(fixed_entries + personalized_entries)
faq_df


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,why was i charged twice this month,Duplicate charges are refunded within 3-5 busi...,double charge refund duplicate,billing
5,can i get an invoice for my last payment,"Yes, invoices are emailed automatically and av...",invoice receipt payment billing,billing


## Q2. Generate and Score a Hypothesis

A scoring function that takes a query string and returns all matching questions ranked by confidence.
Confidence here is the number of overlapping words between the query and an entry's `keywords`
(case-insensitive), and only entries with a score > 0 are returned, sorted highest score first.

In [2]:
def search_faq(query, df):
    """
    Takes a query string and returns all matching FAQ entries ranked by confidence score.
    Confidence score = number of overlapping words between the query and the entry's keywords.
    """
    query_words = set(query.lower().split())
    scores = []
    for _, row in df.iterrows():
        keyword_words = set(str(row["keywords"]).lower().split())
        score = len(query_words & keyword_words)
        scores.append(score)

    result = df.copy()
    result["score"] = scores
    result = result[result["score"] > 0].sort_values("score", ascending=False).reset_index(drop=True)
    return result

# quick demo
search_faq("fee payment", faq_df)


,question,answer,keywords,category,score
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,2
1,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing,1
2,can i get an invoice for my last payment,"Yes, invoices are emailed automatically and av...",invoice receipt payment billing,billing,1


## Q3. `same_category(category_name, df)`

Returns all questions belonging to a given category. Called with the category of one of the
personalized entries from Q1 (`billing`).

In [3]:
def same_category(category_name, df):
    """Return all rows/questions belonging to the given category."""
    return df[df["category"] == category_name]

# Call it using the category of a personalized entry from Q1 ("billing")
billing_questions = same_category("billing", faq_df)
billing_questions


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,why was i charged twice this month,Duplicate charges are refunded within 3-5 busi...,double charge refund duplicate,billing
5,can i get an invoice for my last payment,"Yes, invoices are emailed automatically and av...",invoice receipt payment billing,billing


## Q4. Update a Keyword and Save to CSV

Pick one entry in the knowledge base, ask the user to input a new keyword, add it to that entry's
keywords, then save the entire updated DataFrame to `<roll_number>_faq_data.csv`.

Since this runs non-interactively here, the `input()` call is wrapped so it falls back to a
demo keyword (`"upi"`) when no interactive input is available — but the code is written exactly
as it would be used interactively.

In [4]:
# Pick the entry to update - here, the "how can i pay the fee" entry (index 3)
entry_index_to_update = faq_df[faq_df["question"] == "how can i pay the fee"].index[0]

try:
    new_keyword = input("Enter a new keyword to add to this FAQ entry: ").strip()
    if not new_keyword:
        raise ValueError("empty input")
except Exception:
    new_keyword = "upi"   # fallback demo keyword when run non-interactively
    print(f"(No interactive input detected - using demo keyword: '{new_keyword}')")

faq_df.loc[entry_index_to_update, "keywords"] = (
    faq_df.loc[entry_index_to_update, "keywords"] + " " + new_keyword
)

print("Updated entry:")
print(faq_df.loc[entry_index_to_update])

output_filename = f"{roll_number}_faq_data.csv"
faq_df.to_csv(output_filename, index=False)
print(f"\nSaved updated DataFrame to {output_filename}")


(No interactive input detected - using demo keyword: 'upi')
Updated entry:
question                         how can i pay the fee
answer      You can pay via UPI, card, or net banking.
keywords                       pay payment upi fee upi
category                                       billing
Name: 3, dtype: str

Saved updated DataFrame to 1024160106_faq_data.csv


## Q5. Count FAQ Entries per Category (using `groupby`)

In [5]:
category_counts = faq_df.groupby("category").size().rename("faq_count")
category_counts


category
account    1
billing    4
general    1
Name: faq_count, dtype: int64

## Q6. Handle Ties in the Scoring Function

Modified version of the Q2 scoring function: if two or more entries tie for the highest score,
it prints **all** matching entries instead of silently picking one, so the user can see every
equally good match.

Demonstrated with:
- `"fee"` -> matches both fee-related fixed entries equally -> **tie**
- `"password"` -> matches only one entry -> **no tie**

In [6]:
def search_faq_with_tie_handling(query, df):
    """
    Same scoring logic as search_faq, but explicitly detects and reports ties
    for the highest score instead of silently returning just one best match.
    """
    query_words = set(query.lower().split())
    scores = []
    for _, row in df.iterrows():
        keyword_words = set(str(row["keywords"]).lower().split())
        score = len(query_words & keyword_words)
        scores.append(score)

    result = df.copy()
    result["score"] = scores
    result = result[result["score"] > 0].sort_values("score", ascending=False).reset_index(drop=True)

    if result.empty:
        print(f"No matches found for query: '{query}'")
        return result

    max_score = result["score"].max()
    top_matches = result[result["score"] == max_score]

    if len(top_matches) > 1:
        print(f"Query '{query}': TIE detected — {len(top_matches)} entries "
              f"share the highest score of {max_score}:")
    else:
        print(f"Query '{query}': single best match with score {max_score}:")

    display(top_matches[["question", "answer", "category", "score"]])
    return result

print("--- Demo 1: query that produces a tie ---")
_ = search_faq_with_tie_handling("fee", faq_df)

print("\n--- Demo 2: query that does NOT produce a tie ---")
_ = search_faq_with_tie_handling("password", faq_df)


--- Demo 1: query that produces a tie ---
Query 'fee': TIE detected — 2 entries share the highest score of 1:


,question,answer,category,score
0,what is the annual fee,The annual fee is Rs 500.,billing,1
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1



--- Demo 2: query that does NOT produce a tie ---
Query 'password': single best match with score 1:


,question,answer,category,score
0,how to reset password,Go to Settings > Reset Password.,account,1
